# LLM을 이용한 그래프 디비 설계

자연어 문서를 입력으로 받아 LLM을 이용해 Neo4j 그래프 데이터베이스의 기본 설계를 생한다.

- 문서에 등장하는 기업, 인물, 서비스, 사건, 기술, 매출 정보 등  내용을 분석하여 그래프 DB에서 사용할 노드(Label), 관계(Relationship), 속성(Property)을 도출하고, 이를 바탕으로 Neo4j에 적재할 수 있는 Cypher 쿼리를 생성한다.
- 쿼리 뿐 아니라 설계 스키마와 설계 근거를 설명하는 명세서도 같이 작성하도록 한다.

- **LLM이 생성한 스키마와 쿼리는 초안으로 사용해야 한다.** 이 초안을 바탕으로 데이터 중복, 관계 방향, Label 명명 규칙, 고유 식별자, 검색 목적 등을 사람이 검토하고 도메인에 맞게 정리한 뒤 사용하는 것이 중요하다.


In [4]:
DOCUMENTS = [
    "구글은 1995년 스탠퍼드대에서 만난 래리 페이지와 세르게이 브린이 웹페이지의 링크 구조로 중요도를 판단하는 검색엔진 BackRub을 만들면서 시작됐다. 1998년 Sun 공동창업자 앤디 벡톨샤임의 10만 달러 투자 이후 Google Inc.가 공식 출범했고, 검색을 중심으로 광고, Gmail, Maps, Android, YouTube, Cloud, AI로 확장했다. 현재는 알파벳의 핵심 자회사로, 2025년 알파벳 매출은 4,028억 달러였고 Google Services와 Google Cloud가 성장을 이끌었다.",

    "네이버는 이해진이 삼성SDS 사내벤처에서 출발해 1999년 6월 설립한 한국 인터넷 기업이다. 처음에는 검색 포털 네이버와 어린이 포털 쥬니버로 시작했고, 2000년 통합검색, 2001년 검색광고, 지식iN, 블로그, 카페, 웹툰, 쇼핑, 페이, 클라우드, AI로 영역을 넓혔다. 현재는 한국 대표 플랫폼 기업으로 검색·광고·커머스·핀테크·콘텐츠·클라우드가 주력이다. 2025년 연매출은 12조 350억 원, 영업이익은 2조 2,081억 원이었다.",

    "현재의 카카오는 두 다음과 카카오가 합쳐진 회사다. 다음은 1995년 설립되어 한메일, 다음카페, 미디어다음, 검색으로 성장했고, 카카오는 2006년 아이위랩으로 설립된 뒤 2010년 카카오톡을 내놓으며 모바일 메신저 플랫폼으로 커졌다. 2014년 다음과 카카오는 합병을 발표했고, 이후 회사명은 카카오가 되었다. 현재 주력은 카카오톡 기반 광고·커머스, 모빌리티, 페이, 콘텐츠, 음악, 미디어, AI이다. 2025년 매출은 8조 991억 원, 영업이익은 7,320억 원으로 창사 이래 최대였다.",

    "아마존은 제프 베이조스가 1994년 미국 워싱턴주 벨뷰의 차고에서 시작한 회사다. 1995년 온라인 서점 Amazon.com으로 문을 열었고, 이후 책을 넘어 전자상거래 전반, 마켓플레이스, 물류, Prime, Kindle, 영상 스트리밍, 광고로 확장했다. 2006년 시작한 AWS는 클라우드 컴퓨팅 시장의 핵심 축이 되었다. 현재 아마존은 전자상거래, 클라우드, 광고, 구독, 물류를 모두 가진 세계적 기술·유통 기업이다. 2025년 4분기 순매출은 2,134억 달러였고 AWS 매출은 356억 달러였다.",

    "엔비디아는 젠슨 황, 크리스 말라코스키, 커티스 프리엠이 1993년 창업한 반도체 기업이다. 처음에는 PC 그래픽과 게임용 GPU에 집중했고, 1999년 GeForce 256, 2006년 CUDA를 통해 GPU를 범용 병렬계산과 AI 학습에 쓰는 길을 열었다. 이후 데이터센터, AI 가속기, 네트워킹, 자율주행, 로보틱스, Omniverse 등으로 확장했다. 현재는 생성형 AI와 데이터센터 인프라의 핵심 기업이다. 2026년 1분기 회계연도 2027 매출은 816억 달러, 데이터센터 매출은 752억 달러였다.",
    
    "메타는 마크 저커버그가 더스틴 모스코비츠, 크리스 휴스, 에두아르도 세버린과 함께 2004년 2월 4일 하버드에서 Facebook을 시작하며 출발했다. 처음에는 대학생용 소셜 네트워크였지만 빠르게 일반 대중으로 확장했고, 이후 Instagram, WhatsApp, Oculus를 인수했다. 2021년에는 사명을 Meta로 바꾸고 소셜미디어와 광고, 메신저, 메타버스, AI, AR·VR 기기로 사업을 넓혔다. 현재 주력은 Facebook, Instagram, Messenger, WhatsApp을 포함한 Family of Apps 광고 사업이다. 2025년 총매출은 2,009억 달러였다."
]

## 생성 단위

1. **쿼리와 명세서를 한번에 생성한다.**
   - 문서의 내용이 크지 않을 경우 한번에 작성하도록 한다.
2. **단계적으로 나눠서 처리한다.**
   - 문서의 내용이 많을 경우 하나의 거대한 프롬프트보다 단계적으로 작성할 때 품질이 훨씬 높다.
   1. 노드와 관계 추출 및 CREATE
   2. 제약 조건 설계
   3. INDEX 설계
   4. 명세서 작성
   - 각 단계를 Prompt -> Model -> outputparser의 체인으로 구성한 뒤 연결한다.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

graph_schema_prompt = ChatPromptTemplate.from_template(
    template="""
당신은 Neo4j 그래프 데이터베이스 스키마 설계 전문가이다.

사용자가 제공한 문서를 분석하여 Neo4j에 적합한 그래프 스키마를 설계하고,
그 스키마를 기반으로 Cypher 쿼리를 작성한다.

# 목표

문서의 내용을 바탕으로 다음 작업을 수행한다.

1. 문서에서 주요 개체를 추출하여 노드(Label)를 정의한다.
2. 개체 간 의미 있는 관계를 추출하여 관계 타입(Relationship Type)을 정의한다.
3. 각 노드와 관계에 필요한 속성(Property)을 정의한다.
4. 문서의 내용을 저장하기 위한 Cypher CREATE 쿼리를 작성한다.
5. 각 노드와 관계에 대한 스키마를 표 형태로 출력한다.
6. 데이터 중복 방지와 검색 성능 향상을 위한 제약조건과 인덱스 생성 쿼리를 작성한다.

# 입력 문서

아래 문서를 분석한다.

<문서>
{document}
</문서>

# 작업 지침
- GraphRAG용 스키마를 설계하는 것으로  특히 "LLM이 생성하는 Cypher 쿼리로 탐색하기 쉬운 구조" 여야 한다.

## 1. 노드(Noe) 설계 규칙

- 문서에서 독립적으로 식별 가능한 주요 개체를 노드로 설계한다.
- 다른 개체에서 독립적으로 참조될 수 있어야 한다.
- 그 자체가 직접 검색하거나 탐색하는데 시작점이 될 수있는 개체여야 한다. 
- 사람, 조직, 장소, 상품, 문서, 사건, 개념, 기술, 프로젝트 등은 노드 후보가 될 수 있다.
- 노드 Label은 PascalCase로 작성한다.
  - 예: Person, Company, Product, Project, Document
- 하나의 노드는 최소 하나 이상의 식별 속성을 가져야 한다.
  - 예: name, title, id, code
- 문서에 명시되지 않은 정보는 임의로 만들지 않는다.
- 같은 의미의 개체는 하나의 Label로 통합한다.
  - 예: 기업, 회사, 업체 → Company

## 2. 관계(Relationship) 설계 규칙

- 노드 간 의미 있는 연결을 관계로 설계한다.
- 두 노드 사이에 방향성 있는 어떤 행위, 역할, 연관이어야 한다.
- 관계 타입은 동사 또는 동사구로 작성한다. 
- 모호한 이름은 사용하지 않는다.
- 관계 타입은 대문자와 언더스코어를 사용한다. (대문자 SNAKE_CASE)
  - 예: WORKS_FOR, LOCATED_IN, PRODUCES, PARTICIPATED_IN, BELONGS_TO
- 관계는 방향성을 가지며, 의미상 자연스러운 방향으로 설정한다.
  - 예: (Person)-[:WORKS_FOR]->(Company)
- 관계 자체에 의미 있는 정보가 있으면 관계 속성으로 저장한다.
  - 예: since, role, amount, 
- 단순히 함께 등장했다는 이유만으로 관계를 만들지 않는다.

## 3. 속성(Property) 설계 규칙
- 노드나 관계에 부가적인 정보가 있을 경우 속성으로 설정한다.
- 속성명은 camelCase로 작성한다.
  - 예: name, birthDate, createdAt, totalAmount
- 속성값은 문서에 있는 내용만 사용한다.
- 날짜, 금액, 수량, 상태, 역할 등은 가능한 속성으로 분리한다.
- 긴 설명문은 description, summary, content 등의 속성으로 저장할 수 있다.
- 배열이 필요한 경우 리스트 속성을 사용할 수 있다.
  - 예: tags: ["AI", "GraphDB"]
- 속성 타입 규칙
  - 문자열: String
  - 정수: Integer  
  - 실수: Float
  - 불리언: Boolean
  - 날짜: Date (ISO 형식 문자열)
  - 목록: List

## 4. Cypher CREATE 쿼리 작성 규칙

- 문서에서 추출한 실제 데이터를 기반으로 `CREATE` 쿼리를 작성한다.
- 노드와 관계를 함께 생성하는 Cypher 쿼리를 작성한다.
- 변수명은 소문자로 작성한다.
  - 예: person1, company1, project1
- 문자열 값은 작은따옴표 또는 큰따옴표 중 하나로 일관되게 사용한다.
- 문서에 없는 값은 생성하지 않는다.
- 동일한 노드가 여러 관계에서 사용될 경우 하나의 변수로 재사용한다.
- 여러 개의 독립적인 데이터가 있으면 가독성을 위해 여러 개의 CREATE 문으로 나누어 작성할 수 있다.

## 5. 제약조건 작성 규칙

- 각 주요 노드에는 가능한 경우 고유 식별 속성에 대해 UNIQUE 제약조건을 제안한다.
- 제약조건 이름은 명확하게 작성한다.
  - 예: person_name_unique, company_name_unique
- Neo4j 5.x 이상 문법을 사용한다.

예시:

CREATE CONSTRAINT person_name_unique IF NOT EXISTS
FOR (p:Person)
REQUIRE p.name IS UNIQUE;

## 6. 인덱스 작성 규칙

- 자주 검색될 가능성이 높은 속성에 대해 인덱스를 제안한다.
- 이름, 제목, 코드, 날짜, 상태, 카테고리 등은 인덱스 후보가 될 수 있다.
- UNIQUE 제약조건이 걸린 속성에는 별도 일반 인덱스를 중복으로 만들지 않는다.
- Neo4j 5.x 이상 문법을 사용한다.

예시:

CREATE INDEX movie_title_index IF NOT EXISTS
FOR (m:Movie)
ON (m.title);

## 7. 출력 형식

출력은 markdown으로 작성하며 반드시 아래 내용으로 구성한다.

<출력 구조>
# 1. 추출된 그래프 스키마 요약

문서에서 어떤 기준으로 노드와 관계를 설계 했는지 간단히 설명한다.

---

# 2. 노드 스키마 예

| Label | 설명 | 주요 속성 | 고유 식별 속성 |
|---|---|---|---|
| Person | 사람 | name, role, birthDate | name |

---

# 3. 관계 스키마 예

| 관계 타입 | 시작 노드 | 종료 노드 | 설명 | 관계 속성 |
|---|---|---|---|---|
| WORKS_FOR | Person | Company | 사람이 회사에 소속됨 | role, since |

---

# 4. Cypher CREATE 쿼리

```cypher
// 문서 내용을 기반으로 노드와 관계를 생성하는 쿼리
```

# 5. 제약조건 생성 쿼리
```cypher
// UNIQUE 제약조건 생성 쿼리
```

# 6. Index 생성 쿼리
```cypher
// 검색 성능 향상을 위한 인덱스 생성 쿼리
```

7. 설계 근거

* 왜 해당 개체를 노드로 설계했는지 설명한다.
* 왜 해당 연결을 관계로 설계했는지 설명한다.
* 어떤 속성을 식별자 또는 검색 대상으로 판단했는지 설명한다.
* 문서에 정보가 부족하여 확정하기 어려운 부분이 있다면 명시한다.
</출력 구조>
---

## 8. 주의사항

- 문서에 없는 내용을 추측해서 만들지 않는다.
- Neo4j에서 실행 가능한 Cypher 문법으로 작성한다.
- Label, Relationship Type, Property 이름은 일관된 네이밍 규칙을 따른다.
- 가능한 한 단순하고 명확한 그래프 구조를 우선한다.
- 지나치게 많은 노드와 관계를 만들지 말고, 실제 질의에 유용한 구조를 설계한다.
""")

In [6]:
##########################################
# 지식 그래프로 변환
##########################################
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()


model = ChatOpenAI(model="gpt-5.5")

chain = graph_schema_prompt | model

def convert_to_graph_doc(docs:list[str], labels: list[str]=None, relationship:list[str]=None) -> list[str]:
    """문서들을 받아서 db schema(index, constraint) 와 data 생성(Node, Relationship) 생성 cypher query 만드는 함수

    Args:
        docs (list[str]): 쿼리를 만들 문서들
        labels (list[str], optional): 추출할 Node의 Label들. Defaults to None.
        relationship (list[str], optional): 추출할 관계들. Defaults to None.

    Returns:
        list[str]: 문서별로 생성된 결과를 리스트에 담아서 반환한다.
    """
    graph_docs = []
    for doc in docs:
        result = chain.invoke(
                {
                    "labels": labels if labels else "None",
                    "relationship": relationship if relationship else "None",
                    "document": doc
                }
        )
        graph_docs.append(result)
    
    return graph_docs

In [7]:
graph_docs = convert_to_graph_doc(DOCUMENTS)

In [16]:
from IPython.display import Markdown

# Markdown(graph_docs[5].content)
print(graph_docs[5].content)

# 1. 추출된 그래프 스키마 요약

문서에서 독립적으로 검색·탐색 가능한 개체를 기준으로 `Company`, `Person`, `Product`, `Institution`, `BusinessArea`, `BusinessSegment`, `Event`, `FinancialResult` 노드를 설계했다.

메타의 출발점인 Facebook 시작 사건, 사명 변경, 인수, 사업 확장, 현재 주력 사업, 2025년 총매출 정보를 그래프에서 탐색하기 쉽도록 관계로 분리했다.  
특히 GraphRAG에서 “누가 Facebook을 시작했는가”, “Meta가 인수한 것은 무엇인가”, “Meta의 현재 주력 사업은 무엇인가”, “2025년 매출은 얼마인가”와 같은 질의를 쉽게 처리할 수 있도록 구조화했다.

---

# 2. 노드 스키마 예

| Label | 설명 | 주요 속성 | 고유 식별 속성 |
|---|---|---|---|
| Company | 기업 | name, koreanName, formerName, renamedYear, summary | name |
| Person | 사람 | name | name |
| Product | 제품, 서비스, 앱, 브랜드 | name, category, initialAudience, expandedAudience | name |
| Institution | 기관 또는 장소성 기관 | name | name |
| BusinessArea | 사업 영역 또는 기술 영역 | name | name |
| BusinessSegment | 사업 부문 | name, description | name |
| Event | 사건 | title, eventType, date, year, fromName, toName | title |
| FinancialResult | 재무 실적 | title, year, metricName, amount, currency, unit, amountText | title |

---

# 3. 관계 스키마 예

|

# RAGAS를 이용한 GraphRAG 평가

위에서 만든 `graph_docs`(문서별 스키마 설계 + Cypher 쿼리, 아직 텍스트일 뿐 실제 DB에는 적재되지 않은 상태)를 가지고 다음을 진행한다.

1. `graph_docs`에서 Cypher 쿼리(CREATE, 제약조건, 인덱스)만 추출해 **실제 Neo4j에 적재**
2. 적재된 그래프 위에 `GraphCypherQAChain`으로 **평가 대상 RAG** 구성
3. 원본 `DOCUMENTS`를 재료로 RAGAS `TestsetGenerator`로 **질문/정답 테스트셋** 생성
4. 테스트셋의 각 질문을 RAG에 넣어 응답/검색결과를 채운 뒤 **RAGAS로 평가**

> **주의**: 상단 마크다운에서 언급된 대로, LLM이 생성한 스키마와 쿼리는 초안이다.
> 특히 여러 문서(회사)에 걸쳐 같은 이름의 개체(예: 여러 회사가 공통으로 가진 `AI`, `클라우드` 같은 `BusinessArea`)가
> 문서마다 독립적으로 `CREATE`되면서 **중복 노드**가 생기거나 **UNIQUE 제약조건과 충돌**할 수 있다.
> 아래 적재 함수는 이런 충돌이 나더라도 파이프라인이 끊기지 않도록 statement 단위로 실행하고 실패 내역을 별도로 수집한다.


In [9]:
# !uv pip install ragas rapidfuzz langchain_neo4j neo4j
# 설치 후 커널 재시작


In [10]:
# docker run \
#     --name neo4j \
#     -p 7474:7474 -p 7687:7687 \
#     -d \
#     -e NEO4J_AUTH=neo4j/password \
#     -e NEO4J_PLUGINS='["apoc"]' \
#     neo4j:latest
# 브라우저 콘솔: http://localhost:7474 (neo4j / password 로 로그인)


## 1. `graph_docs` → 실제 Neo4j 적재

`graph_docs[i].content`는 마크다운 텍스트이고, 그 안의 ` ```cypher ... ``` ` 코드블록에
CREATE / 제약조건 / 인덱스 쿼리가 전부 들어있다. 정규식으로 코드블록만 뽑아내고,
세미콜론(;) 기준으로 개별 statement로 쪼개서 하나씩 실행한다.


In [17]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print(
    os.getenv("NEO4J_URI"),
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
    os.getenv("NEO4J_DATABASE")
)

neo4j://127.0.0.1:7687 neo4j asdfghjk neo4j


In [18]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph()

In [19]:
graph.query("RETURN 'hello' ")

[{"'hello'": 'hello'}]

In [20]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

데이터베이스가 초기화되었습니다.


In [21]:
import re
from langchain_core.documents import Document
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship


def extract_cypher_blocks(markdown_text: str) -> "list[str]":
    return re.findall(r"```cypher\s*(.*?)```", markdown_text, flags=re.DOTALL)


def extract_data_blocks(markdown_text: str) -> "list[str]":
    """CREATE 데이터 블록만 남기고 CONSTRAINT/INDEX 블록은 제외 (유령노드 방지)"""
    blocks = extract_cypher_blocks(markdown_text)
    return [b for b in blocks if not re.search(r"CREATE\s+(CONSTRAINT|INDEX)", b, re.IGNORECASE)]


def _split_top_level(s: str, sep: str = ",") -> "list[str]":
    parts, depth, buf, quote = [], 0, "", None
    for ch in s:
        if quote:
            buf += ch
            if ch == quote:
                quote = None
        elif ch in "'\"":
            quote = ch
            buf += ch
        elif ch in "([{":
            depth += 1
            buf += ch
        elif ch in ")]}":
            depth -= 1
            buf += ch
        elif ch == sep and depth == 0:
            parts.append(buf)
            buf = ""
        else:
            buf += ch
    if buf.strip():
        parts.append(buf)
    return parts


def _parse_value(v: str):
    v = v.strip()
    m = re.match(r"^\w+\(\s*(['\"])(.*?)\1\s*\)$", v)
    if m:
        return m.group(2)
    if v.startswith("[") and v.endswith("]"):
        return [_parse_value(x) for x in _split_top_level(v[1:-1])]
    if len(v) >= 2 and v[0] == v[-1] and v[0] in "'\"":
        return v[1:-1]
    if v.lower() in ("true", "false"):
        return v.lower() == "true"
    try:
        return int(v)
    except ValueError:
        try:
            return float(v)
        except ValueError:
            return v


def parse_properties(props_block: str) -> dict:
    props_block = (props_block or "").strip()
    if not (props_block.startswith("{") and props_block.endswith("}")):
        return {}
    props = {}
    for part in _split_top_level(props_block[1:-1]):
        if ":" not in part:
            continue
        key, _, val = part.partition(":")
        props[key.strip()] = _parse_value(val)
    return props


NODE_PATTERN = re.compile(r"\(\s*(\w+)\s*:\s*(\w+)\s*(\{.*?\})?\s*\)")
REL_PATTERN = re.compile(
    r"\(\s*(\w+)\s*\)\s*-\s*\[\s*:\s*(\w+)\s*(\{.*?\})?\s*\]\s*->\s*\(\s*(\w+)\s*\)"
)
IDENTIFYING_KEYS = ["name", "id", "title", "code"]

node_dict = {}
relationships = []

for doc in graph_docs:
    content = doc.content if hasattr(doc, "content") else str(doc)
    var_to_key = {}

    for block in extract_data_blocks(content):
        for var, label, props_str in NODE_PATTERN.findall(block):
            props = parse_properties(props_str)
            id_key = next((props[k] for k in IDENTIFYING_KEYS if k in props), var)
            key = (label, str(id_key))
            if key in node_dict:
                node_dict[key].properties.update(props)
            else:
                node_dict[key] = Node(id=str(id_key), type=label, properties=props)
            var_to_key[var] = key

        for src_var, rel_type, props_str, tgt_var in REL_PATTERN.findall(block):
            src_key, tgt_key = var_to_key.get(src_var), var_to_key.get(tgt_var)
            if src_key is None or tgt_key is None:
                continue
            relationships.append(
                Relationship(
                    source=node_dict[src_key],
                    target=node_dict[tgt_key],
                    type=rel_type,
                    properties=parse_properties(props_str),
                )
            )

nodes = list(node_dict.values())
print(f"노드 {len(nodes)}개, 관계 {len(relationships)}개 파싱 완료")

노드 43개, 관계 0개 파싱 완료


C:\Users\Playdata\AppData\Local\Temp\ipykernel_14916\146063015.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship


In [22]:
# GraphDocument 객체 생성
graph_doc = GraphDocument(
    nodes=nodes,
    relationships=relationships,
    source=Document(page_content="\n\n".join(DOCUMENTS)),  # GraphDocument는 source가 필수 항목
)

# 생성된 GraphDocument를 Neo4j 데이터베이스에 저장
# add_graph_documents 메서드는 GraphDocument 객체 리스트를 받아 데이터베이스에 일괄 저장
graph.add_graph_documents([graph_doc])  # Node, Relationship들을 MERGE 처리.
graph.refresh_schema()
print("그래프 데이터베이스에 저장 완료")


그래프 데이터베이스에 저장 완료


In [23]:
# 노드/관계 개수 확인
counts = graph.query(
    "MATCH (n) RETURN count(n) AS node_count"
)[0]["node_count"]
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN count(r) AS rel_count"
)[0]["rel_count"]

print(f"저장된 노드 수: {counts}")
print(f"저장된 관계 수: {rel_counts}")


저장된 노드 수: 43
저장된 관계 수: 0


In [24]:
print(graph.schema)


Node properties:
Person {name: STRING, id: STRING}
Product {name: STRING, id: STRING}
BusinessArea {name: STRING, id: STRING}
BusinessSegment {name: STRING, id: STRING}
Concept {name: STRING, id: STRING}
Industry {name: STRING, id: STRING}
UseCase {name: STRING, id: STRING}
Institution {name: STRING, id: STRING}
Relationship properties:

The relationships:



## 2. 평가 대상 RAG 구성 (`GraphCypherQAChain`)

질문 → Cypher 생성 → Neo4j 조회 → 자연어 응답 생성을 수행하는 체인이다.
스키마 설계에 썼던 모델(`gpt-5.5`)과는 별개로, Cypher 생성/응답 생성용 모델을 따로 둔다.


In [25]:
from langchain_core.runnables import RunnableLambda
from langchain_neo4j import GraphCypherQAChain

qa_model = ChatOpenAI(model="gpt-4.1")
# GPT-5/o-series reasoning 모델을 쓰려면 temperature 파라미터를 넘기지 않아야 한다.

cypher_chain = GraphCypherQAChain.from_llm(
    llm=qa_model,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,  # Cypher 조회 결과(context) 확인을 위해 필수
    allow_dangerous_requests=True,   # Cypher 실행을 허용하는 플래그 (필수)
    top_k=5,
)


def format_context_from_steps(intermediate_steps: list) -> "list[str]":
    """
    GraphCypherQAChain의 intermediate_steps에서 Cypher 조회 결과를
    RAGAS가 요구하는 list[str] 형태로 직렬화.
    """
    context_rows = intermediate_steps[1]["context"]
    return [str(row) for row in context_rows] if context_rows else []


def run_graph_rag(query: str) -> dict:
    result = cypher_chain.invoke({"query": query})
    return {
        "response": result["result"],
        "retrieved_context": format_context_from_steps(result["intermediate_steps"]),
    }


chain = RunnableLambda(run_graph_rag)


In [26]:
res = chain.invoke("메타는 어떤 기업을 인수했나요?")
res




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: "메타"})-[:INSTITUTION]->(i:Institution) 
RETURN i.name, i.id


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `INSTITUTION` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=33, offset=32>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 32, 'line': 1, 'column': 33}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (p:Person {name: "메타"})-[:INSTITUTION]->(i:Institution) \nRETURN i.name, i.id'


Full Context:
[]

> Finished chain.


{'response': '알려진 바가 없습니다.', 'retrieved_context': []}

## 3. RAGAS 테스트셋 생성

원본 `DOCUMENTS`(6개 회사 소개 문단) 자체가 이미 청크 단위 텍스트 리스트이므로,
Qdrant/Neo4j 적재 여부와 무관하게 `TestsetGenerator`의 재료로 그대로 사용할 수 있다.


In [27]:
# 주피터노트북 환경에서 비동기 처리를 위해 필요 (script(.py)로 실행할 경우는 불필요)
import nest_asyncio
nest_asyncio.apply()


In [28]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

# 참고: RAGAS 0.3.x 초반 버전에서는 gpt-5/o-series 모델 사용 시 temperature 파라미터
# 충돌로 에러가 났었지만, 이후 버전(0.4+)에서는 GPT-5/o-series를 공식 지원하도록 수정됐다.
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 문서에 등장하는 기업(구글, 네이버, 카카오, 아마존, 엔비디아, 메타)에 대해
  사람들이 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법을 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
"""
)

# DOCUMENTS(list[str])를 그대로 재료로 사용
testset = generator.generate_with_chunks(DOCUMENTS, testset_size=10)
eval_df = testset.to_pandas()
eval_df.head(3)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_14916\4056335930.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_14916\4056335930.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


Applying SummaryExtractor:   0%|          | 0/6 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Node ffa03b66-fdb4-4e7f-b2b3-bbdc26d4a8de does not have a summary. Skipping filtering.
Node 2661b08a-0ce3-44b5-b397-1597a25b4b3d does not have a summary. Skipping filtering.
Node 27f4ab74-83aa-4fca-8ff0-cdf5e2685d9c does not have a summary. Skipping filtering.
Node c461ce4e-cd5c-4745-98e5-8116b231b589 does not have a summary. Skipping filtering.
Node 037f3937-957f-4040-9811-3ff1fa882609 does not have a summary. Skipping filtering.
Node ec42c08f-a419-4507-8e60-ba3d10c271d2 does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/6 [00:00<?, ?it/s]

...

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,"구글 클라우드가 뭐고, 구글 회사 성장에 어떻게 영향줬는지 설명해줘요.",[구글은 1995년 스탠퍼드대에서 만난 래리 페이지와 세르게이 브린이 웹페이지의 링...,"구글은 검색을 중심으로 광고, Gmail, Maps, Android, YouTube...",Tech Industry Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,"네이버가 어린이 포털 쥬니버를 통해 어떤 방식으로 서비스 영역을 확장했는지, 그리고...",[네이버는 이해진이 삼성SDS 사내벤처에서 출발해 1999년 6월 설립한 한국 인터...,네이버는 처음에 검색 포털 네이버와 어린이 포털 쥬니버로 시작했습니다. 이후 200...,Korean Tech Industry Analyst,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
2,카카오의 2025년 매출과 영업이익은 얼마였습니까?,[현재의 카카오는 두 다음과 카카오가 합쳐진 회사다. 다음은 1995년 설립되어 한...,"2025년 카카오의 매출은 8조 991억 원이고, 영업이익은 7,320억 원으로 창...",Korean Tech Industry Analyst,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer


## 4. 평가 대상 RAG로 응답/검색결과 채우기


In [29]:
response_list = []
retrieved_context_list = []

for user_input in eval_df["user_input"]:
    resp = chain.invoke(user_input)
    response_list.append(resp["response"])
    retrieved_context_list.append(resp["retrieved_context"])

eval_df["response"] = response_list
eval_df["retrieved_contexts"] = retrieved_context_list
eval_df.head()




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (i:Institution {name: "Google"}), (p:Product {name: "Google Cloud"}) 
RETURN i.name AS Institution, p.name AS Product
Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (n:Institution {name: "네이버"})-[:HAS_PRODUCT]->(p:Product {name: "쥬니버"})-[:BELONGS_TO_BUSINESSAREA]->(ba:BusinessArea)
OPTIONAL MATCH (p)-[:RELATED_TO_BUSINESSSEGMENT]->(bs:BusinessSegment)
OPTIONAL MATCH (p)-[:SUPPORTS_CONCEPT]->(c:Concept)
RETURN n.name AS Institution, p.name AS Product, ba.name AS BusinessArea, collect(DISTINCT bs.name) AS BusinessSegments, collect(DISTINCT c.name) AS SupportedConcepts


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PRODUCT` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=39, offset=38>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 38, 'line': 1, 'column': 39}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n:Institution {name: "네이버"})-[:HAS_PRODUCT]->(p:Product {name: "쥬니버"})-[:BELONGS_TO_BUSINESSAREA]->(ba:BusinessArea)\nOPTIONAL MATCH (p)-[:RELATED_TO_BUSINESSSEGMENT]->(bs:BusinessSegment)\nOPTIONAL MATCH (p)-[:SUPPORTS_CONCEPT]->(c:Concept)\nRETURN n.name AS Institution, p.name AS Product

Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (i:Institution {name: "카카오"}) RETURN i.name, i.id
Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: "제프 베이조스"})-[]-(i:Institution {name: "아마존"})
RETURN p, i
Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (i:Institution {name: "엔비디아"})-[]-(ba:BusinessArea {name: "AI"})
RETURN i, ba
Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (i1:Institution {name: "카카오"})-[:MERGED_WITH]->(i2:Institution {name: "다음"}),
      (i1)-[:FINANCIALS {year: "2025"}]->(f:Concept),
      (i1)-[:MAIN_BUSINESS {year: "2025"}]->(b:BusinessArea),
      (t:Concept {name: "2025년"})
RETURN f.name AS Financials2025, b.name AS MainBusiness2025, t.name AS ThemeYear


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `MERGED_WITH` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=40, offset=39>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 39, 'line': 1, 'column': 40}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (i1:Institution {name: "카카오"})-[:MERGED_WITH]->(i2:Institution {name: "다음"}),\n      (i1)-[:FINANCIALS {year: "2025"}]->(f:Concept),\n      (i1)-[:MAIN_BUSINESS {year: "2025"}]->(b:BusinessArea),\n      (t:Concept {name: "2025년"})\nRETURN f.name AS Financials2025, b.name AS MainBusiness2025

Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `ASSOCIATED_WITH` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=128, offset=127>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 127, 'line': 1, 'column': 128}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (b:BusinessArea {name: "AWS"})<-[:BELONGS_TO]-(p:Product)<-[:GENERATES]-(awsRevenue:Concept {name: "2025 Q4 Revenue"})-[:ASSOCIATED_WITH]->(i:Institution {name: "Amazon"}),\n      (totalRevenue:Concept {name: "2025 Q4 Total Revenue"})-[:ASSOCIATED_WITH]->(i)\nRETURN totalRevenue, aw

Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (b:BusinessArea)<-[:HAS_BUSINESS_AREA]-(p:Product)<-[:GENERATES_REVENUE_FROM]-(i:Institution {name: "카카오"})
WHERE i.id = "2025"
RETURN b.name AS main_business_area, SUM(p.revenue) AS total_revenue, SUM(p.operating_profit) AS total_operating_profit
ORDER BY total_revenue DESC, total_operating_profit DESC
LIMIT 1


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_BUSINESS_AREA` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=27, offset=26>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 26, 'line': 1, 'column': 27}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (b:BusinessArea)<-[:HAS_BUSINESS_AREA]-(p:Product)<-[:GENERATES_REVENUE_FROM]-(i:Institution {name: "카카오"})\nWHERE i.id = "2025"\nRETURN b.name AS main_business_area, SUM(p.revenue) AS total_revenue, SUM(p.operating_profit) AS total_operating_profit\nORDER BY total_revenue DESC, total

Full Context:
[]

> Finished chain.


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_contexts
0,"구글 클라우드가 뭐고, 구글 회사 성장에 어떻게 영향줬는지 설명해줘요.",[구글은 1995년 스탠퍼드대에서 만난 래리 페이지와 세르게이 브린이 웹페이지의 링...,"구글은 검색을 중심으로 광고, Gmail, Maps, Android, YouTube...",Tech Industry Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,모르겠습니다.,[]
1,"네이버가 어린이 포털 쥬니버를 통해 어떤 방식으로 서비스 영역을 확장했는지, 그리고...",[네이버는 이해진이 삼성SDS 사내벤처에서 출발해 1999년 6월 설립한 한국 인터...,네이버는 처음에 검색 포털 네이버와 어린이 포털 쥬니버로 시작했습니다. 이후 200...,Korean Tech Industry Analyst,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer,알려드릴 수 있는 답이 없습니다.,[]
2,카카오의 2025년 매출과 영업이익은 얼마였습니까?,[현재의 카카오는 두 다음과 카카오가 합쳐진 회사다. 다음은 1995년 설립되어 한...,"2025년 카카오의 매출은 8조 991억 원이고, 영업이익은 7,320억 원으로 창...",Korean Tech Industry Analyst,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer,알려진 정보가 없어 답변할 수 없습니다.,[]
3,제프 베이조스는 아마존을 어떻게 시작했습니까?,[아마존은 제프 베이조스가 1994년 미국 워싱턴주 벨뷰의 차고에서 시작한 회사다....,제프 베이조스는 1994년 미국 워싱턴주 벨뷰의 차고에서 아마존을 시작했습니다.,Korean Tech Industry Analyst,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,모르겠습니다.,[]
4,엔비디아가 AI 분야에서 어떤 역할을 하고 있는지 설명해 주세요.,"[엔비디아는 젠슨 황, 크리스 말라코스키, 커티스 프리엠이 1993년 창업한 반도체...","엔비디아는 GPU를 범용 병렬계산과 AI 학습에 활용하는 길을 열었으며, 현재는 생...",Korean Tech Industry Analyst,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer,엔비디아가 AI 분야에서 어떤 역할을 하고 있는지에 대해서는 알지 못합니다.,[]


## 5. RAGAS 평가 실행


In [30]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
)

eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)

# 평가할 때 사용할 LLM, Embedding 모델 (테스트셋 생성용과 동일하게 고정)
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings),
]

eval_result = evaluate(dataset=eval_dataset, metrics=metrics)
eval_result


C:\Users\Playdata\AppData\Local\Temp\ipykernel_14916\3381890477.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_14916\3381890477.py:2: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_14916\3381890477.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\User

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'context_recall': 0.0000, 'llm_context_precision_with_reference': 0.0000, 'faithfulness': 0.1000, 'answer_relevancy': 0.0000}

In [31]:
result_df = eval_result.to_pandas()
result_df


,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,"구글 클라우드가 뭐고, 구글 회사 성장에 어떻게 영향줬는지 설명해줘요.",[],모르겠습니다.,"구글은 검색을 중심으로 광고, Gmail, Maps, Android, YouTube...",0.0,0.0,0.0,0.0
1,"네이버가 어린이 포털 쥬니버를 통해 어떤 방식으로 서비스 영역을 확장했는지, 그리고...",[],알려드릴 수 있는 답이 없습니다.,네이버는 처음에 검색 포털 네이버와 어린이 포털 쥬니버로 시작했습니다. 이후 200...,0.0,0.0,0.0,0.0
2,카카오의 2025년 매출과 영업이익은 얼마였습니까?,[],알려진 정보가 없어 답변할 수 없습니다.,"2025년 카카오의 매출은 8조 991억 원이고, 영업이익은 7,320억 원으로 창...",0.0,0.0,0.0,0.0
3,제프 베이조스는 아마존을 어떻게 시작했습니까?,[],모르겠습니다.,제프 베이조스는 1994년 미국 워싱턴주 벨뷰의 차고에서 아마존을 시작했습니다.,0.0,0.0,1.0,0.0
4,엔비디아가 AI 분야에서 어떤 역할을 하고 있는지 설명해 주세요.,[],엔비디아가 AI 분야에서 어떤 역할을 하고 있는지에 대해서는 알지 못합니다.,"엔비디아는 GPU를 범용 병렬계산과 AI 학습에 활용하는 길을 열었으며, 현재는 생...",0.0,0.0,0.0,0.0
5,"카카오가 다음이랑 합병하고 나서 2025년에 매출이랑 영업이익이 어떻게 됐는지, 그...",[],모르겠습니다.,카카오는 2014년에 다음과 합병해서 회사명이 카카오로 바뀌었어요. 2025년에는 ...,0.0,0.0,0.0,0.0
6,"카카오는 2025년에 어떤 사업 분야를 주력으로 삼았으며, 이 해의 매출과 영업이익...",[],답을 알지 못합니다.,"카카오는 2025년에 카카오톡 기반 광고·커머스, 모빌리티, 페이, 콘텐츠, 음악,...",0.0,0.0,0.0,0.0
7,카카오 2025년 매출 얼마였나?,[],알고 있는 답을 드릴 수 없습니다.,2025년 카카오의 매출은 8조 991억 원이었다.,0.0,0.0,0.0,0.0
8,"아마존의 2025년 4분기 전체 순매출과 AWS 부문 매출은 각각 얼마였으며, 이 ...",[],알려진 바가 없습니다.,"아마존의 2025년 4분기 전체 순매출은 2,134억 달러였고, AWS 부문 매출은...",0.0,0.0,0.0,0.0
9,카카오는 1995년 설립된 다음과 2006년 설립된 카카오가 2014년 합병하여 만...,[],모르겠습니다.,카카오는 1995년 설립된 다음과 2006년 설립된 카카오가 2014년 합병하여 만...,0.0,0.0,0.0,0.0
